# Unified Comparative Evaluation
- **Purpose**: Compare already completed objective evaluations (EXP002, EXP003, EXP004).
- **Structure**: Independent Four-Stem and Two-Stem analytical branches.
- **Evidence-Driven**: Original artifacts are authoritative. No assumptions about schemas or aggregation semantics.


In [1]:
# CELL 2 - RUNTIME AND DEPENDENCY VALIDATION
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
from pathlib import Path
print(f"Pandas: {pd.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")


Pandas: 3.0.3
Numpy: 2.5.2
Matplotlib: 3.11.1


## Phase 1: Four-Stem Comparative Evaluation
Path Configuration for existing Four-Stem Outputs.


In [2]:
# CELL 4 - FOUR-STEM PATH CONFIGURATION
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OBJECTIVE_ROOT = Path(r"D:\IKS_Research\Evaluation\Objective")
EXPERIMENTS = ["EXP002", "EXP003", "EXP004"]
STEMS = ["vocals", "drums", "bass", "other"]
METRICS = ["SDR", "SI-SDR", "SIR", "SAR"]

COMPARATIVE_ROOT = Path(r"D:\IKS_Research\Evaluation\Comparative")
PLOTS_DIR_4 = COMPARATIVE_ROOT / "plots" / "four_stem"
TABLES_DIR_4 = COMPARATIVE_ROOT / "tables" / "four_stem"
PLOTS_DIR_4.mkdir(parents=True, exist_ok=True)
TABLES_DIR_4.mkdir(parents=True, exist_ok=True)
print("Four-Stem paths configured.")


Four-Stem paths configured.


In [3]:
# CELL 5 - FOUR-STEM ARTIFACT DISCOVERY
exp_files_4 = {}
for exp in EXPERIMENTS:
    candidate1 = OBJECTIVE_ROOT / exp / "metrics" / "per_track_metrics.jsonl"
    candidate2 = OBJECTIVE_ROOT / exp / "metrics" / f"{exp}_per_track_metrics.jsonl"
    if candidate1.exists():
        exp_files_4[exp] = candidate1
    elif candidate2.exists():
        exp_files_4[exp] = candidate2
    else:
        raise FileNotFoundError(f"Expected file not found for {exp}")

with open(exp_files_4["EXP002"]) as f:
    sample_line = json.loads(f.readline())
print("Sample record keys:", list(sample_line.keys()))
print("Schema check passed.")


Sample record keys: ['status', 'track', 'track_index', 'sample_rate', 'sources', 'track_average', 'evaluation', 'elapsed_seconds', 'timestamp']
Schema check passed.


In [4]:
# CELL 6 - FOUR-STEM INGESTION
rows_4 = []
for exp, path in exp_files_4.items():
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            if "sources" not in rec: continue
            track_name = rec.get("track", "unknown")
            
            for stem in STEMS:
                if stem in rec["sources"]:
                    metrics_obj = rec["sources"][stem]
                    # Map Four-Stem keys to uniform Two-Stem keys
                    rows_4.append({"experiment": exp, "track": track_name, "stem": stem, "metric": "SDR", "value": metrics_obj.get("SDR_dB", np.nan)})
                    rows_4.append({"experiment": exp, "track": track_name, "stem": stem, "metric": "SI-SDR", "value": metrics_obj.get("SI_SDR_dB", np.nan)})
                    rows_4.append({"experiment": exp, "track": track_name, "stem": stem, "metric": "SIR", "value": metrics_obj.get("SIR_dB", np.nan)})
                    rows_4.append({"experiment": exp, "track": track_name, "stem": stem, "metric": "SAR", "value": metrics_obj.get("SAR_dB", np.nan)})

df_4 = pd.DataFrame(rows_4)
df_4 = df_4.dropna(subset=["value"])
print(f"Ingested {len(df_4)} metric values across {df_4['experiment'].nunique()} experiments.")


Ingested 2400 metric values across 3 experiments.


In [5]:
# CELL 7 - FOUR-STEM VALIDATION
counts_4 = df_4.groupby(["experiment", "stem"])["track"].nunique().unstack()
print("Track counts per experiment/stem (expect 50 for each):")
print(counts_4)
problems_4 = []
for exp in EXPERIMENTS:
    for stem in STEMS:
        n = counts_4.loc[exp, stem] if exp in counts_4.index and stem in counts_4.columns else 0
        if n != 50:
            problems_4.append(f"{exp}/{stem}: found {n}, expected 50")
if problems_4:
    print("WARNING -- count mismatch, investigate before trusting results:")
    for p in problems_4:
        print("  -", p)
else:
    print("All experiment/stem combinations have 50 tracks. Proceeding.")


Track counts per experiment/stem (expect 50 for each):
stem        bass  drums  other  vocals
experiment                            
EXP002        50     50     50      50
EXP003        50     50     50      50
EXP004        50     50     50      50
All experiment/stem combinations have 50 tracks. Proceeding.


In [6]:
# CELL 8 - FOUR-STEM SUMMARY TABLE
summary_4 = df_4.groupby(["experiment", "stem", "metric"])["value"].agg(["mean", "std"]).reset_index()
summary_4["mean_std"] = summary_4.apply(lambda r: f"{r['mean']:.2f} \u00b1 {r['std']:.2f}", axis=1)
pivot_table_4 = summary_4.pivot_table(index="experiment", columns=["metric", "stem"],
                                   values="mean_std", aggfunc="first").reindex(EXPERIMENTS)

# Reorder columns
pivot_table_4 = pivot_table_4.reindex(columns=pd.MultiIndex.from_product([METRICS, STEMS], names=['metric', 'stem']))
pivot_table_4.to_csv(TABLES_DIR_4 / "four_stem_summary_table.csv")
print(pivot_table_4)


metric              SDR                                              SI-SDR  \
stem             vocals        drums         bass        other       vocals   
experiment                                                                    
EXP002      8.59 ± 2.89  9.83 ± 3.75  8.65 ± 4.52  6.22 ± 2.57  7.82 ± 3.83   
EXP003      8.25 ± 3.41  9.67 ± 3.85  8.39 ± 4.53  5.99 ± 2.48  7.38 ± 4.48   
EXP004      8.67 ± 2.85  9.95 ± 3.61  8.62 ± 4.47  6.17 ± 2.63  7.80 ± 3.83   

metric                                                      SIR                \
stem              drums         bass        other        vocals         drums   
experiment                                                                      
EXP002      9.48 ± 3.79  7.85 ± 5.15  4.73 ± 3.92  18.24 ± 4.13  20.66 ± 5.50   
EXP003      9.17 ± 3.98  7.45 ± 5.27  4.40 ± 3.94  18.03 ± 5.07  20.48 ± 5.50   
EXP004      9.51 ± 3.80  7.76 ± 5.14  4.71 ± 3.90  18.31 ± 4.16  20.79 ± 5.53   

metric                                

In [7]:
# CELL 9 - FOUR-STEM PDF EXPORT
fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")
tbl = ax.table(cellText=pivot_table_4.values, rowLabels=pivot_table_4.index,
                colLabels=[f"{m}\n{s}" for m, s in pivot_table_4.columns],
                loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 2)
plt.tight_layout()
plt.savefig(TABLES_DIR_4 / "four_stem_summary_table.pdf", bbox_inches="tight", dpi=200)
plt.close()
print("Summary table saved.")


Summary table saved.


In [8]:
# CELL 10 - FOUR-STEM GROUPED BAR CHARTS
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
bar_width = 0.2
x = np.arange(len(EXPERIMENTS))
colors = {"vocals": "#1f77b4", "drums": "#ff7f0e", "bass": "#2ca02c", "other": "#d62728"}

for ax, metric in zip(axes, METRICS):
    for i, stem in enumerate(STEMS):
        means, stds = [], []
        for exp in EXPERIMENTS:
            vals = df_4[(df_4.experiment == exp) & (df_4.stem == stem) & (df_4.metric == metric)]["value"]
            means.append(vals.mean())
            stds.append(vals.std())
        offset = (i - 1.5) * bar_width
        ax.bar(x + offset, means, bar_width, yerr=stds, capsize=3,
               label=stem.capitalize(), color=colors[stem])
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(EXPERIMENTS)
    ax.set_ylabel(f"{metric} (dB)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].legend(loc="best", fontsize=9)
fig.suptitle("Four-Stem Objective Evaluation: Stem Performance by Experiment", fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR_4 / "grouped_bar_charts.png", dpi=200, bbox_inches="tight")
plt.close()
print("Grouped bar chart saved.")


Grouped bar chart saved.


In [9]:
# CELL 11 - FOUR-STEM HEATMAP
import seaborn as sns
heat_data_4 = summary_4.pivot_table(index="experiment", columns=["metric", "stem"], values="mean").reindex(EXPERIMENTS)
heat_data_4 = heat_data_4.reindex(columns=pd.MultiIndex.from_product([METRICS, STEMS]))

col_min, col_max = heat_data_4.min(axis=0), heat_data_4.max(axis=0)
col_range = (col_max - col_min).replace(0, 1)
normalized = (heat_data_4 - col_min) / col_range

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(normalized.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(heat_data_4.columns)))
ax.set_xticklabels([f"{m}\n{s}" for m, s in heat_data_4.columns], fontsize=8, rotation=45, ha='right')
ax.set_yticks(range(len(heat_data_4.index)))
ax.set_yticklabels(heat_data_4.index)
for i in range(heat_data_4.shape[0]):
    for j in range(heat_data_4.shape[1]):
        ax.text(j, i, f"{heat_data_4.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Relative rank within each column")
ax.set_title("Four-Stem Metric Comparison Across Experiments")
plt.tight_layout()
plt.savefig(PLOTS_DIR_4 / "heatmap.png", dpi=200, bbox_inches="tight")
plt.close()
print("Heatmap saved.")


Heatmap saved.


In [10]:
# CELL 12 - FOUR-STEM (skipped)
pass


In [11]:
# CELL 13 - FOUR-STEM (skipped)
pass


In [12]:
# CELL 14 - FOUR-STEM (skipped)
pass


In [13]:
# CELL 15 - FOUR-STEM FINAL REPORT
print("=== FOUR-STEM FINAL REPORT ===")
print("Generated four-stem metrics successfully.")
for f in [TABLES_DIR_4 / "four_stem_summary_table.csv", TABLES_DIR_4 / "four_stem_summary_table.pdf",
          PLOTS_DIR_4 / "grouped_bar_charts.png", PLOTS_DIR_4 / "heatmap.png"]:
    print(f, "EXISTS" if f.exists() else "MISSING")


=== FOUR-STEM FINAL REPORT ===
Generated four-stem metrics successfully.
D:\IKS_Research\Evaluation\Comparative\tables\four_stem\four_stem_summary_table.csv EXISTS
D:\IKS_Research\Evaluation\Comparative\tables\four_stem\four_stem_summary_table.pdf EXISTS
D:\IKS_Research\Evaluation\Comparative\plots\four_stem\grouped_bar_charts.png EXISTS
D:\IKS_Research\Evaluation\Comparative\plots\four_stem\heatmap.png EXISTS


## Phase 2: Two-Stem Comparative Evaluation
Path Configuration for new Two-Stem Outputs.


In [14]:
# CELL 17 - TWO-STEM PATH CONFIGURATION
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

TWO_SOURCE_ROOT = Path(r"D:\IKS_Research\Evaluation\Two_Source")
EXPERIMENTS = ["EXP002", "EXP003", "EXP004"]
STEMS = ["vocals", "accompaniment"]
METRICS = ["SDR", "SI-SDR", "SIR", "SAR"]

COMPARATIVE_ROOT = Path(r"D:\IKS_Research\Evaluation\Comparative")
PLOTS_DIR = COMPARATIVE_ROOT / "plots" / "two_stem"
TABLES_DIR = COMPARATIVE_ROOT / "tables" / "two_stem"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print("Two-Stem paths configured.")


Two-Stem paths configured.


In [15]:
# CELL 18 - TWO-STEM ARTIFACT DISCOVERY
exp_files = {}
for exp in EXPERIMENTS:
    candidate = TWO_SOURCE_ROOT / exp / "metrics" / "two_source_results.jsonl"
    if not candidate.exists():
        raise FileNotFoundError(f"Expected file not found for {exp}: {candidate}")
    exp_files[exp] = candidate

with open(exp_files["EXP002"]) as f:
    sample_line = json.loads(f.readline())
print("Sample record keys:", list(sample_line.keys()))
NESTED_METRICS = "metrics" in sample_line and isinstance(sample_line["metrics"], dict)
print("Nested metrics object found:", NESTED_METRICS)

missing = [m for m in METRICS if
           (NESTED_METRICS and m not in sample_line.get("metrics", {})) or
           (not NESTED_METRICS and m not in sample_line)]
if missing:
    raise KeyError(f"Expected metrics {missing} not found in: {sample_line}")
print("Schema check passed.")


Sample record keys: ['experiment', 'track', 'source', 'metrics']
Nested metrics object found: True
Schema check passed.


In [16]:
# CELL 19 - TWO-STEM INGESTION
rows = []
for exp, path in exp_files.items():
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            metric_source = rec["metrics"] if NESTED_METRICS else rec
            for m in METRICS:
                rows.append({
                    "experiment": exp,
                    "track": rec.get("track", rec.get("track_id", "unknown")),
                    "stem": rec["source"],
                    "metric": m,
                    "value": metric_source[m],
                })
df = pd.DataFrame(rows)
print(f"Ingested {len(df)} metric values across {df['experiment'].nunique()} experiments.")


Ingested 1200 metric values across 3 experiments.


In [17]:
# CELL 20 - TWO-STEM VALIDATION
counts = df.groupby(["experiment", "stem"])["track"].nunique().unstack()
print("Track counts per experiment/stem (expect 50/50 for each):")
print(counts)
problems = []
for exp in EXPERIMENTS:
    for stem in STEMS:
        n = counts.loc[exp, stem] if exp in counts.index and stem in counts.columns else 0
        if n != 50:
            problems.append(f"{exp}/{stem}: found {n}, expected 50")
if problems:
    print("WARNING -- count mismatch, investigate before trusting results:")
    for p in problems:
        print("  -", p)
else:
    print("All experiment/stem combinations have 50 tracks. Proceeding.")


Track counts per experiment/stem (expect 50/50 for each):
stem        accompaniment  vocals
experiment                       
EXP002                 50      50
EXP003                 50      50
EXP004                 50      50
All experiment/stem combinations have 50 tracks. Proceeding.


In [18]:
# CELL 21 - TWO-STEM SUMMARY TABLE
summary = df.groupby(["experiment", "stem", "metric"])["value"].agg(["mean", "std"]).reset_index()
summary["mean_std"] = summary.apply(lambda r: f"{r['mean']:.2f} \u00b1 {r['std']:.2f}", axis=1)
pivot_table = summary.pivot_table(index="experiment", columns=["metric", "stem"],
                                   values="mean_std", aggfunc="first").reindex(EXPERIMENTS)
pivot_table.to_csv(TABLES_DIR / "two_stem_summary_table.csv")
print(pivot_table)


metric               SAR                        SDR               \
stem       accompaniment       vocals accompaniment       vocals   
experiment                                                         
EXP002      15.83 ± 4.30  8.93 ± 3.45  14.24 ± 4.42  8.59 ± 2.89   
EXP003      15.59 ± 4.35  8.56 ± 3.74  14.18 ± 4.31  8.25 ± 3.41   
EXP004      15.87 ± 4.41  8.91 ± 3.47  14.37 ± 4.25  8.67 ± 2.85   

metric            SI-SDR                        SIR                
stem       accompaniment       vocals accompaniment        vocals  
experiment                                                         
EXP002      14.24 ± 4.22  7.83 ± 3.84  22.61 ± 4.49  19.40 ± 4.08  
EXP003      14.03 ± 4.19  7.41 ± 4.45  22.20 ± 4.63  19.20 ± 4.91  
EXP004      14.29 ± 4.25  7.81 ± 3.84  22.53 ± 4.50  19.45 ± 4.10  


In [24]:
# CELL 22 - TWO-STEM PDF EXPORT
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.axis("off")
tbl = ax.table(cellText=pivot_table.values, rowLabels=pivot_table.index,
                colLabels=[f"{m}\n{s}" for m, s in pivot_table.columns],
                loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.8)
plt.tight_layout()
plt.savefig(TABLES_DIR / "two_stem_summary_table.pdf", bbox_inches="tight", dpi=200)
plt.savefig(TABLES_DIR / "two_stem_summary_table.png", bbox_inches="tight", dpi=200)
plt.close()
print("Summary table saved.")


Summary table saved.


In [22]:
# CELL 23 - TWO-STEM GROUPED BAR CHARTS
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
bar_width = 0.35
x = np.arange(len(EXPERIMENTS))
colors = {"vocals": "#1f77b4", "accompaniment": "#d62728"}

for ax, metric in zip(axes, METRICS):
    for i, stem in enumerate(STEMS):
        means, stds = [], []
        for exp in EXPERIMENTS:
            vals = df[(df.experiment == exp) & (df.stem == stem) & (df.metric == metric)]["value"]
            means.append(vals.mean())
            stds.append(vals.std())
        offset = (i - 0.5) * bar_width
        ax.bar(x + offset, means, bar_width, yerr=stds, capsize=4,
               label=stem.capitalize(), color=colors[stem])
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(EXPERIMENTS)
    ax.set_ylabel(f"{metric} (dB)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Two-Stem Objective Evaluation: Vocals vs. Accompaniment, by Experiment", fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "grouped_bar_charts.png", dpi=200, bbox_inches="tight")
plt.close()
print("Grouped bar chart saved.")


Grouped bar chart saved.


In [26]:
# CELL 24 - TWO-STEM HEATMAP
heat_data = summary.pivot_table(index="experiment", columns=["metric", "stem"], values="mean").reindex(EXPERIMENTS)
col_min, col_max = heat_data.min(axis=0), heat_data.max(axis=0)
col_range = (col_max - col_min).replace(0, 1)
normalized = (heat_data - col_min) / col_range

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(normalized.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels([f"{m}\n{s}" for m, s in heat_data.columns], fontsize=8)
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels(heat_data.index)
for i in range(heat_data.shape[0]):
    for j in range(heat_data.shape[1]):
        ax.text(j, i, f"{heat_data.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Relative rank within each metric column")
ax.set_title("Two-Stem Metric Comparison Across Experiments\n(cell text = actual dB value; color = relative rank within its own column)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "heatmap.png", dpi=200, bbox_inches="tight")
plt.close()
print("Heatmap saved.")


Heatmap saved.


In [29]:
# CELL 25 - TWO-STEM (skipped)
pass


In [30]:
# CELL 26 - TWO-STEM (skipped)
pass


In [31]:
# CELL 27 - TWO-STEM (skipped)
pass
